In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Si, SEPD

This example demonstrates a Rietveld refinement of Si crystal
structure using time-of-flight neutron powder diffraction data from
SEPD at Argonne.

It also shows how to switch calculation engine and peak profile type.

## 🛠️ Import Library

In [ ]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [ ]:
structure = StructureFactory.from_scratch(name='si')

### Set Space Group

In [ ]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.coord_system_code = '2'

### Set Unit Cell

In [ ]:
structure.cell.length_a = 5.431

### Set Atom Sites

In [ ]:
structure.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0.125,
    fract_y=0.125,
    fract_z=0.125,
    adp_iso=0.5,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their
parameters, and link the structures defined in the previous step.

### Download Data

In [ ]:
data_path = download_data('meas-si-sepd', destination='data')

### Create Experiment

In [ ]:
expt = ExperimentFactory.from_data_path(
    name='sepd',
    data_path=data_path,
    beam_mode='time-of-flight',
)

### Set Instrument

In [ ]:
expt.instrument.setup_twotheta_bank = 144.845
expt.instrument.calib_d_to_tof_offset = 0.0
expt.instrument.calib_d_to_tof_linear = 7476.91
expt.instrument.calib_d_to_tof_quadratic = -1.54

### Set Peak Profile

In [ ]:
expt.peak.show_supported()

In [ ]:
expt.peak.type = 'jorgensen-von-dreele'

In [ ]:
expt.peak.broad_gauss_sigma_0 = 3.0148
expt.peak.broad_gauss_sigma_1 = 33.3451
expt.peak.broad_lorentz_gamma_1 = 2.5489
expt.peak.decay_beta_0 = 0.04221
expt.peak.decay_beta_1 = 0.00946
expt.peak.rise_alpha_1 = 0.5971

In [ ]:
expt.peak.cutoff_fwhm = 10

### Set Background

In [ ]:
expt.background.auto_estimate()

### Set Linked Structures

In [ ]:
expt.linked_structures.create(structure_id='si', scale=600.0)

## 📦 Define Project

The project object is used to manage the structure, experiment, and
analysis.

### Create Project

In [ ]:
project = Project(name='si_sepd')

### Add Structure

In [ ]:
project.structures.add(structure)

### Add Experiment

In [ ]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

### Display Structure

In [ ]:
project.display.structure(struct_name='si')

### Display Pattern

In [ ]:
project.display.pattern(expt_name='sepd')
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 1/4

Set parameters to be refined.

In [ ]:
structure.cell.length_a.free = True

expt.linked_structures['si'].scale.free = True
expt.instrument.calib_d_to_tof_offset.free = True

Show free parameters after selection.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.minimizer.type = 'bumps (lm)'

In [ ]:
project.analysis.fit()
project.display.fit.results()

#### Display Pattern

In [ ]:
project.display.pattern(expt_name='sepd')

In [ ]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 2/4

Set more parameters to be refined.

In [ ]:
for point in expt.background:
    point.intensity.free = True

Show free parameters after selection.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()

#### Display Pattern

In [ ]:
project.display.pattern(expt_name='sepd')

In [ ]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 3/4

Fix background points.

In [ ]:
for point in expt.background:
    point.intensity.free = False

Set more parameters to be refined.

In [ ]:
expt.peak.broad_gauss_sigma_0.free = True
expt.peak.broad_gauss_sigma_1.free = True
expt.peak.broad_lorentz_gamma_1.free = True

Show free parameters after selection.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()

#### Display Pattern

In [ ]:
project.display.pattern(expt_name='sepd')

In [ ]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 4/4

Set more parameters to be refined.

In [ ]:
structure.atom_sites['Si'].adp_iso.free = True

expt.peak.decay_beta_0.free = True
expt.peak.decay_beta_1.free = True
expt.peak.rise_alpha_1.free = True

Show free parameters after selection.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()

#### Display Correlations

In [ ]:
project.display.fit.correlations()

#### Display Pattern

In [ ]:
project.display.pattern(expt_name='sepd')

In [ ]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

In [ ]:
project.display.pattern(expt_name='sepd', x='d_spacing')

## 💾 Save Project

In [ ]:
project.save_as(dir_path='projects/refine-si-sepd')